### Import Dependencies

In [1]:
from groq import Groq
import instructor
from qdrant_client import QdrantClient

from pydantic import BaseModel, Field



### RAG PipeLine

In [3]:
class RAGGenerationResponse(BaseModel):
    answer: str=Field(..., description="The answer to the question")
    reasoning_trace: str=Field(..., description="The reasoning trace for the answer")

In [4]:
import os

from langchain_huggingface import HuggingFaceEndpointEmbeddings

QDRANT_URL = "http://localhost:6333"
# Environment variables
HF_API_TOKEN = os.environ.get("HF_API_TOKEN")
QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")

# Model Configuration
HF_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GROQ_LLM_MODEL = "qwen/qwen3-32b"

model = "sentence-transformers/all-MiniLM-L6-v2"
hf = HuggingFaceEndpointEmbeddings(
            model=model,
            huggingfacehub_api_token=os.environ["HF_API_TOKEN"],
        )

client = instructor.from_provider("groq/qwen/qwen3-32b")

c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from groq import Groq
from qdrant_client import QdrantClient
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json


def _mean_pool_embedding(raw_embedding):
    if not raw_embedding:
        raise ValueError("Hugging Face API returned an empty embedding.")

    if isinstance(raw_embedding[0], (int, float)):
        return [float(value) for value in raw_embedding]

    if isinstance(raw_embedding[0], list):
        token_count = len(raw_embedding)
        vector_size = len(raw_embedding[0])
        pooled = [0.0] * vector_size

        for token_vector in raw_embedding:
            if len(token_vector) != vector_size:
                raise ValueError("Inconsistent token vector dimensions in HF embedding response.")
            for idx, value in enumerate(token_vector):
                pooled[idx] += float(value)

        return [value / token_count for value in pooled]

    raise ValueError("Unexpected Hugging Face embedding response format.")

def get_embedding(text, model_name: str | None = None):
    """
    Embed text using HuggingFace Embeddings API.
    Model: sentence-transformers/all-MiniLM-L6-v2
    """
    selected_model = model_name or HF_EMBEDDING_MODEL
    endpoint = (
        "https://router.huggingface.co/hf-inference/models/"
        f"{selected_model}/pipeline/feature-extraction"
    )
    payload = json.dumps(
        {
            "inputs": text,
            "normalize": True,
        }
    ).encode("utf-8")

    headers = {"Content-Type": "application/json"}
    if HF_API_TOKEN:
        headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

    request = Request(endpoint, data=payload, headers=headers, method="POST")

    try:
        with urlopen(request, timeout=60) as response:
            response_data = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        message = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Hugging Face embedding API request failed ({exc.code}): {message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

    if isinstance(response_data, dict) and response_data.get("error"):
        raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

    if isinstance(response_data, list) and len(response_data) == 1:
        return _mean_pool_embedding(response_data[0])
    

    return _mean_pool_embedding(response_data)

def retrieve_data(query, qdrant_client, top_k=5):
    """
    Retrieve similar documents from Qdrant vector database.
    
    Args:
        query (str): User question
        qdrant_client: Qdrant client instance
        top_k (int): Number of documents to retrieve
    
    Returns:
        dict: Retrieved context with IDs, descriptions, ratings, and similarity scores
    """
    query_embedding = get_embedding(query)

    search_result = qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for search_result in search_result.points:
        retrieved_context_ids.append(search_result.payload["parent_asin"])
        retrieved_context.append(search_result.payload["description"])
        retrieved_context_ratings.append(search_result.payload["average_rating"])
        similarity_scores.append(search_result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

def process_context(context):
    """Format retrieved context for LLM prompt."""
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    """Build the prompt for the LLM."""
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in the stock.

    You will be given a question and a list of context.

    Instructions:
    -You need to answer the question based on the provided context only.
    -never use the word context and refer to it as the available products.

    Context:    
    {preprocessed_context}

    Question: {question}
""" 
    return prompt

# Initialize Groq client
#client = Groq()

def generate_answer(prompt):
    """
    Generate answer using Groq LLM.
    
    Model: qwen/qwen3-32b
    
    Args:
        prompt (str): The formatted prompt with context and question
        
    Returns:
        str: Generated answer from the LLM
    """
    completion,raw_response = client.create_with_completion(
        messages=[
        {
            "role": "system",
            "content": prompt
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden",
        temperature=0,
        response_model=RAGGenerationResponse,
    )
    return completion

def rag_pipeline(query,qdrant_client, top_k=5):
    """
    Complete RAG Pipeline.
    
    Models Used:
    - Embedding: HuggingFace (sentence-transformers/all-MiniLM-L6-v2)
    - LLM: Groq (qwen/qwen3-32b)
    - Vector DB: Qdrant
    
    Args:
        query (str): User question
        top_k (int): Number of context documents to retrieve
        
    Returns:
        dict: {
            "answer": str - Generated answer,
            "question": str - Original question,
            "retrieved_context_ids": list - IDs of retrieved documents,
            "retrieved_context": list - Content of retrieved documents,
            "similarity_scores": list - Relevance scores
        }
    """
    #qdrant_client = QdrantClient(url=QDRANT_URL)

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_result = {
        "datamodel":answer,
        "answer": answer.answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [6]:
qdrant_client=QdrantClient(url="http://localhost:6333")

In [10]:
output = rag_pipeline("What are the best rock albums i can get?",qdrant_client)

In [13]:
output

{'datamodel': RAGGenerationResponse(answer="Here are some top rock albums from our collection: 1. **Now That's What I Call Punk & New Wave** (4.7) - A 34-track vinyl compilation featuring The Clash, U2, and The Cure, capturing the punk/new wave era. 2. **I Am The Moon: IV. Farewell** (4.7) - Tedeschi Trucks Band’s 24-song epic inspired by pandemic-era drama. 3. **Totem** (4.7) - Soulfly’s brutal metal album with guest appearances, produced by Max Cavalera. 4. **MERCY** (4.5) - John Cale’s genre-defying album with collaborators like Weyes Blood and Sylvan Esso. 5. **Felt Mountain** (4.8) - Goldfrapp’s Mercury Prize-nominated debut reissued on gold vinyl. Each offers unique styles from classic rock to experimental metal.", reasoning_trace='Selected albums based on high ratings (4.5+), genre diversity, and unique features like iconic artists (U2, Max Cavalera), historical significance (punk/new wave compilation), and reissues with added value (Gold vinyl, new sleeve notes). Prioritized al

### RAG pipeline with Grounding Context

In [14]:
class RAGUsedContext(BaseModel):
    id: str=Field(..., description="The ID of the item used to answer the question")
    description: str=Field(..., description="Short description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str=Field(..., description="The answer to the question")
    references: list[RAGUsedContext]=Field(..., description="List of Items used to answer the question")


In [29]:



def _mean_pool_embedding(raw_embedding):
    if not raw_embedding:
        raise ValueError("Hugging Face API returned an empty embedding.")

    if isinstance(raw_embedding[0], (int, float)):
        return [float(value) for value in raw_embedding]

    if isinstance(raw_embedding[0], list):
        token_count = len(raw_embedding)
        vector_size = len(raw_embedding[0])
        pooled = [0.0] * vector_size

        for token_vector in raw_embedding:
            if len(token_vector) != vector_size:
                raise ValueError("Inconsistent token vector dimensions in HF embedding response.")
            for idx, value in enumerate(token_vector):
                pooled[idx] += float(value)

        return [value / token_count for value in pooled]

    raise ValueError("Unexpected Hugging Face embedding response format.")

def get_embedding(text, model_name: str | None = None):
    """
    Embed text using HuggingFace Embeddings API.
    Model: sentence-transformers/all-MiniLM-L6-v2
    """
    selected_model = model_name or HF_EMBEDDING_MODEL
    endpoint = (
        "https://router.huggingface.co/hf-inference/models/"
        f"{selected_model}/pipeline/feature-extraction"
    )
    payload = json.dumps(
        {
            "inputs": text,
            "normalize": True,
        }
    ).encode("utf-8")

    headers = {"Content-Type": "application/json"}
    if HF_API_TOKEN:
        headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

    request = Request(endpoint, data=payload, headers=headers, method="POST")

    try:
        with urlopen(request, timeout=60) as response:
            response_data = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        message = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Hugging Face embedding API request failed ({exc.code}): {message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

    if isinstance(response_data, dict) and response_data.get("error"):
        raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

    if isinstance(response_data, list) and len(response_data) == 1:
        return _mean_pool_embedding(response_data[0])
    

    return _mean_pool_embedding(response_data)

def retrieve_data(query, qdrant_client, top_k=5):
    """
    Retrieve similar documents from Qdrant vector database.
    
    Args:
        query (str): User question
        qdrant_client: Qdrant client instance
        top_k (int): Number of documents to retrieve
    
    Returns:
        dict: Retrieved context with IDs, descriptions, ratings, and similarity scores
    """
    query_embedding = get_embedding(query)

    search_result = qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for search_result in search_result.points:
        retrieved_context_ids.append(search_result.payload["parent_asin"])
        retrieved_context.append(search_result.payload["description"])
        retrieved_context_ratings.append(search_result.payload["average_rating"])
        similarity_scores.append(search_result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

def process_context(context):
    """Format retrieved context for LLM prompt."""
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    """Build the prompt for the LLM."""

    prompt = f"""You are a shopping assistant that can answer questions about the products in stock. 
    
    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Never show the product IDs in the answer.
    - As an output you need to provide:

    * The answer to the question based on the provided context.
    * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
    * Short description (1-2 sentences) of the item based on the description provided in the context.

    - The short description should have the name of the item.
    - The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

    Context:
    {preprocessed_context}

    Question:
    {question}
    """ 
    return prompt

# Initialize Groq client
#client = Groq()

def generate_answer(prompt):
    """
    Generate answer using Groq LLM.
    
    Model: qwen/qwen3-32b
    
    Args:
        prompt (str): The formatted prompt with context and question
        
    Returns:
        str: Generated answer from the LLM
    """
    completion,raw_response = client.create_with_completion(
        messages=[
        {
            "role": "system",
            "content": prompt
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden",
        temperature=0,
        response_model=RAGGenerationResponse,
    )
    return completion

def rag_pipeline(query,qdrant_client, top_k=5):
    """
    Complete RAG Pipeline.
    
    Models Used:
    - Embedding: HuggingFace (sentence-transformers/all-MiniLM-L6-v2)
    - LLM: Groq (qwen/qwen3-32b)
    - Vector DB: Qdrant
    
    Args:
        query (str): User question
        top_k (int): Number of context documents to retrieve
        
    Returns:
        dict: {
            "answer": str - Generated answer,
            "question": str - Original question,
            "retrieved_context_ids": list - IDs of retrieved documents,
            "retrieved_context": list - Content of retrieved documents,
            "similarity_scores": list - Relevance scores
        }
    """
    #qdrant_client = QdrantClient(url=QDRANT_URL)

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_result = {
        "original_output":answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [30]:
output = rag_pipeline("What are the best metal albums i can get?",qdrant_client,top_k=10)

In [31]:
output

{'original_output': RAGGenerationResponse(answer='- **72 Seasons by Metallica**: Metallica’s 12th studio album, released on April 14, 2023. Produced by Greg Fidelman with Hetfield & Ulrich, it spans over 77 minutes with 12 tracks. This is their first full-length album since 2016’s *Hardwired…To Self-Destruct*. The concept explores childhood experiences shaping identity, as explained by James Hetfield. Rated 4.7.\n\n- **Totem by Soulfly**: Soulfly’s 2022 twelfth album, produced by Max Cavalera and Arthur Rizk. It features guest appearances from John Powers, Chris Ulsh, and John Tardy. The album blends blackened thrash and death metal, continuing Max Cavalera’s legacy from Sepultura. It includes nods to classic Soulfly tracks like *Ritual* and *Archangel*. Rated 4.7.', references=[RAGUsedContext(id='B0BNJQMG9K', description='72 Seasons by Metallica, a 2023 studio album exploring childhood identity through 12 tracks.'), RAGUsedContext(id='B09ZLQW9ZT', description='Totem by Soulfly, a 2022

In [32]:
print(output["answer"])

- **72 Seasons by Metallica**: Metallica’s 12th studio album, released on April 14, 2023. Produced by Greg Fidelman with Hetfield & Ulrich, it spans over 77 minutes with 12 tracks. This is their first full-length album since 2016’s *Hardwired…To Self-Destruct*. The concept explores childhood experiences shaping identity, as explained by James Hetfield. Rated 4.7.

- **Totem by Soulfly**: Soulfly’s 2022 twelfth album, produced by Max Cavalera and Arthur Rizk. It features guest appearances from John Powers, Chris Ulsh, and John Tardy. The album blends blackened thrash and death metal, continuing Max Cavalera’s legacy from Sepultura. It includes nods to classic Soulfly tracks like *Ritual* and *Archangel*. Rated 4.7.
